# Scipy Documentation:
[ttest_1samp()](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_1samp.html)

[stats.ttest_ind()](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html)

[stats.ttest_rel()](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html)

One sample t test

Is the average customer satisfaction score significantly different from 4.0?

Imagine a company claims that its average customer satisfaction is 4.0 out of 5. We collected feedback from 30 customers and want to test whether our sample mean is statistically different from 4.0.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# ---------------------------------------------------------
# 1. Create synthetic dataset
# ---------------------------------------------------------

df = pd.DataFrame({
    "customer_id": range(1, 31),
    "satisfaction_score": [
        4.2, 3.8, 4.5, 4.1, 4.0,
        3.9, 4.3, 4.6, 4.1, 3.7,
        4.4, 4.2, 4.0, 3.8, 4.5,
        4.1, 4.3, 4.2, 3.9, 4.6,
        4.4, 4.1, 3.6, 4.0, 4.2,
        4.5, 4.3, 4.1, 3.9, 4.7
    ]
})

print(df.head())

   customer_id  satisfaction_score
0            1                 4.2
1            2                 3.8
2            3                 4.5
3            4                 4.1
4            5                 4.0


In [2]:
print("Descriptive statistics:")
print(df["satisfaction_score"].describe())

sample_mean = df["satisfaction_score"].mean()
claimed_mean = 4.0

print("Sample mean:", sample_mean)
print("Claimed mean:", claimed_mean)
print("Difference:", sample_mean - claimed_mean)

Descriptive statistics:
count    30.000000
mean      4.166667
std       0.280803
min       3.600000
25%       4.000000
50%       4.150000
75%       4.375000
max       4.700000
Name: satisfaction_score, dtype: float64
Sample mean: 4.166666666666667
Claimed mean: 4.0
Difference: 0.16666666666666696


In [3]:
print("Missing values:", df["satisfaction_score"].isna().sum())

Missing values: 0


In [4]:
shapiro_stat, shapiro_p = stats.shapiro(df["satisfaction_score"])

print("Shapiro-Wilk statistic:", shapiro_stat)
print("Shapiro-Wilk p-value:", shapiro_p)

if shapiro_p < 0.05:
    print("The data may not be normally distributed.")
else:
    print("No strong evidence against normality.")

Shapiro-Wilk statistic: 0.9798276663380285
Shapiro-Wilk p-value: 0.8209524482255122
No strong evidence against normality.


In [5]:
Q1 = df["satisfaction_score"].quantile(0.25)
Q3 = df["satisfaction_score"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["satisfaction_score"] < lower_bound) |
    (df["satisfaction_score"] > upper_bound)
]

print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Outliers:")
print(outliers)

Lower bound: 3.4375
Upper bound: 4.9375
Outliers:
Empty DataFrame
Columns: [customer_id, satisfaction_score]
Index: []


In [ ]:
# stats.ttest_1samp(a, popmean, alternative="two-sided")

In [6]:
result = stats.ttest_1samp(
    a=df["satisfaction_score"],
    popmean=4.0,
    alternative="two-sided",
    nan_policy="omit"
)

print("t-statistic:", result.statistic)
print("p-value:", result.pvalue)
print("degrees of freedom:", result.df)

t-statistic: 3.2509249636885755
p-value: 0.0029126314905435114
degrees of freedom: 29


In [8]:
ci = result.confidence_interval(confidence_level=0.95)

print("\n95% confidence interval:")
print("Lower:", ci.low)
print("Upper:", ci.high)



95% confidence interval:
Lower: 4.061812937088314
Upper: 4.27152039624502


In [ ]:
ci = result.confidence_interval(confidence_level=0.95)

print("\n95% confidence interval:")
print("Lower:", ci.low)
print("Upper:", ci.high)


In [7]:
alpha = 0.05

if result.pvalue < alpha:
    print("Reject H0: The average satisfaction score is significantly different from 4.0.")
else:
    print("Fail to reject H0: There is not enough evidence that the average satisfaction score differs from 4.0.")

Reject H0: The average satisfaction score is significantly different from 4.0.


Independent two sample t test

A company wants to know whether employees who completed a new training program have different productivity scores compared with employees who did not complete the training.

In this scario we have : 

Trained employees
Untrained employees

Is there a significant difference in productivity scores between trained and untrained employees?

This is a two-sided question because we are asking whether there is a difference, not specifically whether trained employees are higher.

In [9]:
import pandas as pd
from scipy import stats

# ---------------------------------------------------------
# 1. Create synthetic dataset
# ---------------------------------------------------------

df = pd.DataFrame({
    "employee_id": range(1, 41),

    "group": [
        "Trained", "Trained", "Trained", "Trained", "Trained",
        "Trained", "Trained", "Trained", "Trained", "Trained",
        "Trained", "Trained", "Trained", "Trained", "Trained",
        "Trained", "Trained", "Trained", "Trained", "Trained",

        "Untrained", "Untrained", "Untrained", "Untrained", "Untrained",
        "Untrained", "Untrained", "Untrained", "Untrained", "Untrained",
        "Untrained", "Untrained", "Untrained", "Untrained", "Untrained",
        "Untrained", "Untrained", "Untrained", "Untrained", "Untrained"
    ],

    "productivity_score": [
        # Trained group
        78, 82, 85, 88, 90,
        79, 84, 86, 91, 87,
        83, 89, 92, 80, 85,
        88, 86, 90, 84, 95,

        # Untrained group
        70, 74, 76, 79, 73,
        77, 75, 72, 80, 78,
        69, 76, 74, 81, 75,
        77, 73, 79, 71, 82
    ]
})

print(df.head())

   employee_id    group  productivity_score
0            1  Trained                  78
1            2  Trained                  82
2            3  Trained                  85
3            4  Trained                  88
4            5  Trained                  90


In [10]:
trained = df[df["group"] == "Trained"]["productivity_score"]
untrained = df[df["group"] == "Untrained"]["productivity_score"]

print("Trained group:")
print(trained.describe())

print("\nUntrained group:")
print(untrained.describe())

Trained group:
count    20.000000
mean     86.100000
std       4.447353
min      78.000000
25%      83.750000
50%      86.000000
75%      89.250000
max      95.000000
Name: productivity_score, dtype: float64

Untrained group:
count    20.000000
mean     75.550000
std       3.634267
min      69.000000
25%      73.000000
50%      75.500000
75%      78.250000
max      82.000000
Name: productivity_score, dtype: float64


In [11]:
print("Missing values in trained group:", trained.isna().sum())
print("Missing values in untrained group:", untrained.isna().sum())

Missing values in trained group: 0
Missing values in untrained group: 0


In [12]:
trained_shapiro = stats.shapiro(trained)
untrained_shapiro = stats.shapiro(untrained)

print("Trained group Shapiro-Wilk:")
print("Statistic:", trained_shapiro.statistic)
print("p-value:", trained_shapiro.pvalue)

print("\nUntrained group Shapiro-Wilk:")
print("Statistic:", untrained_shapiro.statistic)
print("p-value:", untrained_shapiro.pvalue)

Trained group Shapiro-Wilk:
Statistic: 0.9872330727332022
p-value: 0.9920604166050505

Untrained group Shapiro-Wilk:
Statistic: 0.9830317718801566
p-value: 0.9669947066304174


In [13]:
if trained_shapiro.pvalue > 0.05 and untrained_shapiro.pvalue > 0.05:
    print("No strong evidence against normality in both groups.")
else:
    print("At least one group may not be normally distributed.")

No strong evidence against normality in both groups.


In [14]:
# Classic independent t-test assumes both groups have similar variance.
levene_result = stats.levene(trained, untrained)

print("Levene test for equal variances:")
print("Statistic:", levene_result.statistic)
print("p-value:", levene_result.pvalue)

Levene test for equal variances:
Statistic: 0.5527771098821829
p-value: 0.4617575968193597


In [15]:
if levene_result.pvalue > 0.05:
    print("Variances are not significantly different. Equal variance assumption is acceptable.")
else:
    print("Variances may be different. Welch's t-test is safer.")

Variances are not significantly different. Equal variance assumption is acceptable.


so we are supposed to use : equal_var=True

extra:equal_var=False runs Welch’s t-test.

In [16]:
result = stats.ttest_ind(
    a=trained,
    b=untrained,
    equal_var=True,
    alternative="two-sided",
    nan_policy="omit"
)

print("Independent two-sample t-test:")
print("t-statistic:", result.statistic)
print("p-value:", result.pvalue)
print("degrees of freedom:", result.df)

Independent two-sample t-test:
t-statistic: 8.214801743832085
p-value: 5.966047570161203e-10
degrees of freedom: 38.0


a=trained
→ first independent group

b=untrained
→ second independent group

equal_var=True 
→ use when variances in both group are close/same

equal_var=False
→ use Welch’s t-test
→ safer when group variances may not be equal

alternative="two-sided"
→ test whether the two group means are different in either direction

nan_policy="omit"
→ ignore missing values if they exist (In our case we did not have any missing value)

In [17]:
ci = result.confidence_interval(confidence_level=0.95)

print("95% confidence interval for mean difference:")
print("Lower:", ci.low)
print("Upper:", ci.high)

95% confidence interval for mean difference:
Lower: 7.950137082394347
Upper: 13.149862917605647


In [18]:
alpha = 0.05

if result.pvalue < alpha:
    print("Reject H0: There is a significant difference between trained and untrained employees.")
else:
    print("Fail to reject H0: There is not enough evidence of a significant difference.")

Reject H0: There is a significant difference between trained and untrained employees.


Paired sample t test

This is the parametric brother of the Wilcoxon signed-rank test. Basically scenario of "after before".
Same people/items measured twice, and we test whether the mean difference is statistically significant.

Scenario:
A company introduced a new task-management system. The same employees were measured before and after the system was introduced.

Did employees’ productivity scores change after the new task-management system?

In [19]:
import pandas as pd
from scipy import stats

df = pd.DataFrame({
    "employee_id": range(1, 26),
    "productivity_before": [
        62, 65, 67, 70, 72,
        68, 74, 71, 69, 75,
        73, 66, 78, 76, 72,
        70, 69, 74, 77, 71,
        68, 73, 75, 70, 72
    ],
    "productivity_after": [
        66, 69, 70, 74, 76,
        71, 78, 75, 72, 80,
        76, 69, 82, 79, 75,
        73, 73, 78, 81, 74,
        71, 76, 79, 73, 75
    ]
})

df["difference"] = df["productivity_after"] - df["productivity_before"]

print("Dataset:")
print(df)

print("\nDescriptive statistics:")
print(df[["productivity_before", "productivity_after", "difference"]].describe())

print("\nChange summary:")
print("Increased:", (df["difference"] > 0).sum())
print("No change:", (df["difference"] == 0).sum())
print("Decreased:", (df["difference"] < 0).sum())

print("\nMissing values:")
print(df[["productivity_before", "productivity_after"]].isna().sum())

shapiro_result = stats.shapiro(df["difference"])

print("\nShapiro-Wilk normality test for differences:")
print("Statistic:", shapiro_result.statistic)
print("p-value:", shapiro_result.pvalue)

Q1 = df["difference"].quantile(0.25)
Q3 = df["difference"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["difference"] < lower_bound) |
    (df["difference"] > upper_bound)
]

print("\nOutlier check for differences:")
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print(outliers)

result = stats.ttest_rel(
    a=df["productivity_after"],
    b=df["productivity_before"],
    alternative="two-sided",
    nan_policy="omit"
)

print("\nPaired-sample t-test:")
print("t-statistic:", result.statistic)
print("p-value:", result.pvalue)
print("degrees of freedom:", result.df)

ci = result.confidence_interval(confidence_level=0.95)

print("\n95% confidence interval for mean paired difference:")
print("Lower:", ci.low)
print("Upper:", ci.high)

alpha = 0.05

if result.pvalue < alpha:
    print("\nConclusion: Productivity changed significantly after the new task-management system.")
else:
    print("\nConclusion: There is not enough evidence of a significant productivity change.")

Dataset:
    employee_id  productivity_before  productivity_after  difference
0             1                   62                  66           4
1             2                   65                  69           4
2             3                   67                  70           3
3             4                   70                  74           4
4             5                   72                  76           4
5             6                   68                  71           3
6             7                   74                  78           4
7             8                   71                  75           4
8             9                   69                  72           3
9            10                   75                  80           5
10           11                   73                  76           3
11           12                   66                  69           3
12           13                   78                  82           4
13           14          

# TO SUM UP:

stats.ttest_1samp()
1samp = 1 sample vs 1 fixed number

stats.ttest_1samp(
    a=sample,
    popmean=known_value,
    alternative="two-sided",
    nan_policy="omit"
)

stats.ttest_ind()
ind   = independent groups

stats.ttest_ind(
    a=group_1,
    b=group_2,
    equal_var=False,
    alternative="two-sided",
    nan_policy="omit"
)

stats.ttest_rel()
rel   = related/repeated/paired data

stats.ttest_rel(
    a=after,
    b=before,
    alternative="two-sided",
    nan_policy="omit"
)

One sample?
→ stats.ttest_1samp(sample, popmean=value)

Two different groups?
→ stats.ttest_ind(group1, group2, equal_var=False)

Same group twice?
→ stats.ttest_rel(after, before)